<a href="https://colab.research.google.com/github/warrensuca/chinese-recipe-api/blob/main/omnivores_cookbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Omnivore's Cookbook Scraper
This notebook scrapes recipe data from the Omnivore's Cookbook Website: https://www.chinasichuanfood.com/

For each recipe, we collect:

- Name
- Ingredients Full List
- Ingredients Names List
  - Used in clustering, very difficult to parse from full list
- Procedure
- Nutrition Facts

The final dataset is stored as a Pandas DataFrame and exported to CSV.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display
import time
print("setup complete")

setup complete


In [ ]:
session = requests.Session()

session.headers.update({
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print("Session initialized")

Session initialized


#Scrape individual recipe URL
Exploration. Later, we will turn this into a function

In [22]:

url = "https://omnivorescookbook.com/black-pepper-chicken/"
headings = [
      "Name",
      "Ingredients_List",
      "Ingredients_Names",
      "Procedure",
      "Nutrition_Facts"
  ]

response = session.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

recipe_data = {
    "Name": url.split("/")[-2] #getting title is an additional call, just clean later
}
print(recipe_data["Name"])
print('\n')

ingredients_list_raw = soup.find_all("li", class_ = "wprm-recipe-ingredient")
ingredients_list = []

for ingredient in ingredients_list_raw:
  ingredients_list.append(ingredient.get_text()) #remove a bullet point svg

print(ingredients_list)
print('\n')
recipe_data["Ingredients_List"] = ingredients_list

ingredients_names_raw = soup.find_all("span", class_ = "wprm-recipe-ingredient-name")
ingredients_names = []

for ingredient_name in ingredients_names_raw:
  ingredients_names.append(ingredient_name.get_text())

print(ingredients_names)
print('\n')
recipe_data["Ingredients_Names"] = ingredients_names

procedure_list_raw = soup.find_all("li", class_ = "wprm-recipe-instruction")
procedure_list = []

for procedure in procedure_list_raw:
  procedure_list.append(procedure.get_text())

print(procedure_list)
print('\n')
recipe_data["Procedure"] = procedure_list

nutrition_facts_list_raw = soup.find_all("span", class_ = "wprm-nutrition-label-text-nutrition-container")
nutrition_facts = []
for nutrition_fact in nutrition_facts_list_raw:
  #print(nutrition_fact.get_text())
  nutrition_facts.append(nutrition_fact.get_text())

print(nutrition_facts)

recipe_data["Nutrition_Facts"] = nutrition_facts

black-pepper-chicken


['1 lb chicken breasts (or thighs) , sliced against the grain into 1/4” (5-mm) thick pieces', '1 tablespoon light soy sauce (or soy sauce)', '1 tablespoon Shaoxing wine (or dry sherry)', '1 tablespoon cornstarch', '1/2 cup chicken broth', '2 tablespoons light soy sauce (or soy sauce)', '2 tablespoons Shaoxing wine (or dry sherry)', '2 teaspoons dark soy sauce (or soy sauce)', '1 tablespoon cornstarch', '1 1/2 tablespoons sugar', '2 teaspoons coarsely ground black pepper', '1/8 teaspoon salt', '2 tablespoons peanut oil (or vegetable oil)', '1 tablespoon minced ginger', '2 cloves garlic , minced', '1/2 white onion , chopped', '2 bell peppers , chopped (I used mixed colors)']


['chicken breasts (or thighs)', 'light soy sauce', 'Shaoxing wine', 'cornstarch', 'chicken broth', 'light soy sauce', 'Shaoxing wine', 'dark soy sauce', 'cornstarch', 'sugar', 'coarsely ground black pepper', 'salt', 'peanut oil', 'minced ginger', 'garlic', 'white onion', 'bell peppers']


['C

#Create function based off exploration
This function will be used on each recipe of the site

In [23]:
def scrape_recipe(url: str):


  response = session.get(url, timeout=10)
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")

  recipe_data = {
      "Name": url.split("/")[-2] #getting title is an additional call, just clean later
  }


  ingredients_list_raw = soup.find_all("li", class_ = "wprm-recipe-ingredient")
  ingredients_list = []

  for ingredient in ingredients_list_raw:
    ingredients_list.append(ingredient.get_text()) #remove a bullet point svg


  recipe_data["Ingredients_List"] = ingredients_list


  ingredients_names_raw = soup.find_all("span", class_ = "wprm-recipe-ingredient-name")
  ingredients_names = []

  for ingredient_name in ingredients_names_raw:
    ingredients_names.append(ingredient_name.get_text())

  recipe_data["Ingredients_Names"] = ingredients_names


  procedure_list_raw = soup.find_all("li", class_ = "wprm-recipe-instruction")
  procedure_list = []

  for procedure in procedure_list_raw:
    procedure_list.append(procedure.get_text())



  recipe_data["Procedure"] = procedure_list

  nutrition_facts_list_raw = soup.find_all("span", class_ = "wprm-nutrition-label-text-nutrition-container")
  nutrition_facts = []
  for nutrition_fact in nutrition_facts_list_raw:
    #print(nutrition_fact.get_text())
    nutrition_facts.append(nutrition_fact.get_text())

  recipe_data["Nutrition_Facts"] = nutrition_facts

  return recipe_data

In [ ]:
recipe_data = scrape_recipe("https://omnivorescookbook.com/black-pepper-chicken/")

recipe_data

{'Name': 'black-pepper-chicken',
 'Ingredients_List': ['1 lb chicken breasts (or thighs) , sliced against the grain into 1/4” (5-mm) thick pieces',
  '1 tablespoon light soy sauce (or soy sauce)',
  '1 tablespoon Shaoxing wine (or dry sherry)',
  '1 tablespoon cornstarch',
  '1/2 cup chicken broth',
  '2 tablespoons light soy sauce (or soy sauce)',
  '2 tablespoons Shaoxing wine (or dry sherry)',
  '2 teaspoons dark soy sauce (or soy sauce)',
  '1 tablespoon cornstarch',
  '1 1/2 tablespoons sugar',
  '2 teaspoons coarsely ground black pepper',
  '1/8 teaspoon salt',
  '2 tablespoons peanut oil (or vegetable oil)',
  '1 tablespoon minced ginger',
  '2 cloves garlic , minced',
  '1/2 white onion , chopped',
  '2 bell peppers , chopped (I used mixed colors)'],
 'Ingredients_Names': ['chicken breasts (or thighs)',
  'light soy sauce',
  'Shaoxing wine',
  'cornstarch',
  'chicken broth',
  'light soy sauce',
  'Shaoxing wine',
  'dark soy sauce',
  'cornstarch',
  'sugar',
  'coarsely gro

#Scrape single page
Exploration

In [ ]:
page = 3

response = session.get("https://omnivorescookbook.com/category/recipe/?_paged={}".format(page), timeout=10)
response.raise_for_status()

data = []

soup = BeautifulSoup(response.text, "html.parser")

recipes = soup.find_all(class_ = "gridtitle")


for recipe in recipes[:5]:

  data.append(scrape_recipe(recipe.a.get("href")))
data

[{'Name': 'bok-choy-with-oyster-sauce',
  'Ingredients_List': ['1 lb baby bok choy',
   '1 tablespoon peanut oil (or vegetable oil)',
   '2 dried chili peppers',
   '2 teaspoons ginger , minced',
   '1/2 teaspoon sugar',
   '1 tablespoon soy sauce',
   '2 tablespoons oyster sauce'],
  'Ingredients_Names': ['baby bok choy',
   'peanut oil',
   'dried chili peppers',
   'ginger',
   'sugar',
   'soy sauce',
   'oyster sauce'],
  'Procedure': ['Cut off the ends of baby bok choy and discard. Separate the larger leaves of baby bok choy. Soak in cold water for 2 minutes, then rinse to remove any dust. Cut the stems into 1/2” pieces and the leaves into 2” pieces. Separate the whites and greens. Halve the cores.',
   'Heat oil in a wok over medium-high heat until smoking. Add chili peppers and ginger. Stir a few times to release fragrance.',
   'Add the baby bok choy whites and turn to high heat. Stir a few times to coat evenly with oil, spread them out and sear until the white part just start

In [14]:
def scrape_page(page_num: int):

  response = session.get("https://omnivorescookbook.com/category/recipe/?_paged={}".format(page_num), timeout=10)
  response.raise_for_status()

  data = []

  soup = BeautifulSoup(response.text, "html.parser")

  recipes = soup.find_all(class_ = "gridtitle")


  for recipe in recipes:

    data.append(scrape_recipe(recipe.a.get("href")))

  return data

In [15]:
columns = [
      "Name",
      "Ingredients_List",
      "Ingredients_Names",
      "Procedure",
      "Nutrition_Facts"
  ]

temp_out = []

for page_num in range(1, 3):
  data = scrape_page(page_num)
  temp_out.extend(data)
#print(out)
df = pd.DataFrame(temp_out, columns=columns)

print(f"Total recipes scraped: {len(df)}")

display(df.head())

Total recipes scraped: 80


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
0,thai-fish-cakes,"[1 pound white fish fillet , cut into 1-inch (...","[white fish fillet, red curry paste, large egg...","[Fish cake:, Combine the fish, curry paste, eg...","[Serving: 1of the 6 servings, Calories: 308kca..."
1,budae-jjigae,"[1 tablespoon peanut oil (or vegetable oil), 1...","[peanut oil, ground beef, green onions, garlic...",[Heat oil in a 4-quart dutch oven (or heavy po...,"[Serving: 4servings, Calories: 522kcal, Carboh..."
2,drunken-chicken,[2 bone-in skin-on chicken leg quarters (inclu...,"[bone-in skin-on chicken leg quarters, salt, g...",[Combine the salt and Sichuan peppercorns in a...,"[Serving: 1serving, Calories: 99kcal, Carbohyd..."
3,matcha-tiramisu,[2 tablespoons ceremonial grade matcha and ext...,"[ceremonial grade matcha, very hot water, masc...",[To make the matcha soak: Sift matcha into a m...,"[Serving: 1serving, Calories: 224kcal, Carbohy..."
4,clams-in-black-bean-sauce,"[2 lbs manila clams, Salt , for soaking the cl...","[manila clams, Salt, vegetable oil, Shaoxing W...",[Place the clams in a large bowl. Add 8 cups c...,"[Serving: 1serving, Calories: 318kcal, Carbohy..."


#Scrape the entire site!

In [16]:
pages = 18

out = []

for page_num in range(1, pages + 1):



    try:
      data = scrape_page(page_num)
      out.extend(data)

    except Exception as e:

      print(f"Error on page {page_num}: {e}")
      break
    print(f"page number: {page_num}")
    time.sleep(4) #be nice to the site, too aggresive can cause an exception


page number: 1
page number: 2
page number: 3
page number: 4
page number: 5
page number: 6
page number: 7
page number: 8
page number: 9
page number: 10
page number: 11
page number: 12
page number: 13
page number: 14
page number: 15
page number: 16
page number: 17
page number: 18


#Store Data in Pandas df
We can then get some insights and then export to CSV file

In [17]:
columns = [
      "Name",
      "Ingredients_List",
      "Ingredients_Names",
      "Procedure",
      "Nutrition_Facts"
  ]

df = pd.DataFrame(out, columns=columns)

print(f"Total recipes scraped: {len(df)}")

display(df.head())

Total recipes scraped: 688


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
0,thai-fish-cakes,"[1 pound white fish fillet , cut into 1-inch (...","[white fish fillet, red curry paste, large egg...","[Fish cake:, Combine the fish, curry paste, eg...","[Serving: 1of the 6 servings, Calories: 308kca..."
1,budae-jjigae,"[1 tablespoon peanut oil (or vegetable oil), 1...","[peanut oil, ground beef, green onions, garlic...",[Heat oil in a 4-quart dutch oven (or heavy po...,"[Serving: 4servings, Calories: 522kcal, Carboh..."
2,drunken-chicken,[2 bone-in skin-on chicken leg quarters (inclu...,"[bone-in skin-on chicken leg quarters, salt, g...",[Combine the salt and Sichuan peppercorns in a...,"[Serving: 1serving, Calories: 99kcal, Carbohyd..."
3,matcha-tiramisu,[2 tablespoons ceremonial grade matcha and ext...,"[ceremonial grade matcha, very hot water, masc...",[To make the matcha soak: Sift matcha into a m...,"[Serving: 1serving, Calories: 224kcal, Carbohy..."
4,clams-in-black-bean-sauce,"[2 lbs manila clams, Salt , for soaking the cl...","[manila clams, Salt, vegetable oil, Shaoxing W...",[Place the clams in a large bowl. Add 8 cups c...,"[Serving: 1serving, Calories: 318kcal, Carbohy..."


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 688 entries, 0 to 687
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               688 non-null    object
 1   Ingredients_List   688 non-null    object
 2   Ingredients_Names  688 non-null    object
 3   Procedure          688 non-null    object
 4   Nutrition_Facts    688 non-null    object
dtypes: object(5)
memory usage: 27.0+ KB


In [19]:
df = df.loc[df.astype(str).drop_duplicates().index]
df

,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
0,thai-fish-cakes,"[1 pound white fish fillet , cut into 1-inch (...","[white fish fillet, red curry paste, large egg...","[Fish cake:, Combine the fish, curry paste, eg...","[Serving: 1of the 6 servings, Calories: 308kca..."
1,budae-jjigae,"[1 tablespoon peanut oil (or vegetable oil), 1...","[peanut oil, ground beef, green onions, garlic...",[Heat oil in a 4-quart dutch oven (or heavy po...,"[Serving: 4servings, Calories: 522kcal, Carboh..."
2,drunken-chicken,[2 bone-in skin-on chicken leg quarters (inclu...,"[bone-in skin-on chicken leg quarters, salt, g...",[Combine the salt and Sichuan peppercorns in a...,"[Serving: 1serving, Calories: 99kcal, Carbohyd..."
3,matcha-tiramisu,[2 tablespoons ceremonial grade matcha and ext...,"[ceremonial grade matcha, very hot water, masc...",[To make the matcha soak: Sift matcha into a m...,"[Serving: 1serving, Calories: 224kcal, Carbohy..."
4,clams-in-black-bean-sauce,"[2 lbs manila clams, Salt , for soaking the cl...","[manila clams, Salt, vegetable oil, Shaoxing W...",[Place the clams in a large bowl. Add 8 cups c...,"[Serving: 1serving, Calories: 318kcal, Carbohy..."
...,...,...,...,...,...
683,spinach-salad,"[6 tablespoons olive oil, 2 tablespoons rice v...","[olive oil, rice vinegar, shallot, honey, toas...",[Wash spinach thoroughly with running water. R...,[]
684,baked-costa-rican-style-fish-with-black-beans-...,"[300 g fish fillet tilapia, bass, or catfish, ...","[fish fillet, freshly squeezed orange juice, f...",[Add fish and all the ingredients for marinate...,[]
685,creamy-chicken-potato-salad,"[1 lb (450 g) red potatoes (*Footnote 1), 1 ta...","[red potatoes, plus 1/2 teaspoon salt, boneles...",[Prepare a steamer by adding water to the pot....,"[Serving: 1serving, Calories: 295kcal, Carbohy..."
686,pork-stock,"[1.4 kg (3 pounds) pork leg bones, 1 thumbnail...","[pork leg bones, thumbnail ginger, large onion]",[Put pork bones in a large pot and add water u...,[]


#Save Data to CSV

In [20]:
output_file = "omnivores_cookbook_recipes.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Output file: {output_file}")

Output file: omnivores_cookbook_recipes.csv
